# DeblurDiff Motivation Probe

This notebook generates the first-stage motivation evidence for efficient DeblurDiff:

- step-count quality/runtime sweeps on `imgs/input` and `imgs/target`
- internal LKPN/EAC and denoising-dynamics heatmaps
- paper-style grids and CSV exports
- optional `pyiqa` sanity checks in a separate IQA environment

Run this on the CUDA DeblurDiff environment with the released checkpoint available.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import torch
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.research.motivation_utils import (
    DEFAULT_NEG_PROMPT,
    DEFAULT_STEPS,
    build_pipeline,
    check_pyiqa_environment,
    ensure_output_dirs,
    list_image_pairs,
    load_restored_by_step,
    load_rgb,
    plot_internal_maps,
    plot_runtime_quality,
    plot_step_grid,
    rows_to_dataframe,
    run_pyiqa_pair,
    run_step_sweep,
)

print(f"Repo root: {ROOT}")
print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## Config

For a quick smoke test, set `steps_list = [50, 10]` and `max_images = 1`. For the full motivation run, keep the defaults.

In [ ]:
model_path = ROOT / "checkpoint" / "model.pth"
input_dir = ROOT / "imgs" / "input"
target_dir = ROOT / "imgs" / "target"
output_dir = ROOT / "research_outputs" / "motivation_imgs"

device = "cuda" if torch.cuda.is_available() else "cpu"
steps_list = DEFAULT_STEPS
seed = 231
max_images = None

strength = 1.0
tiled = False
tile_size = 512
tile_stride = 256
pos_prompt = ""
neg_prompt = DEFAULT_NEG_PROMPT
cfg_scale = 1.0
better_start = False

dirs = ensure_output_dirs(output_dir)
print(dirs)
print(f"model_path exists: {model_path.exists()} -> {model_path}")

## Load Model And Image Pairs

DeblurDiff's LKPN/EAC path requires CUDA/CuPy. If `device` is CPU here, do not run the inference sweep.

In [ ]:
pairs = list_image_pairs(input_dir, target_dir)
if max_images is not None:
    pairs = pairs[:max_images]
pairs

In [ ]:
if not model_path.exists():
    raise FileNotFoundError(f"Set model_path to the released checkpoint: {model_path}")
if device != "cuda":
    raise RuntimeError("DeblurDiff LKPN/EAC inference should be run on CUDA.")

pipeline = build_pipeline(model_path, device=device)
print("Pipeline ready")

## Run DeblurDiff Step Sweep

This cell writes restored images, trace tensors, and `step_sweep_metrics.csv` under `research_outputs/motivation_imgs/`.

In [ ]:
rows = run_step_sweep(
    pipeline=pipeline,
    image_pairs=pairs,
    output_dir=output_dir,
    steps_list=steps_list,
    seed=seed,
    strength=strength,
    tiled=tiled,
    tile_size=tile_size,
    tile_stride=tile_stride,
    pos_prompt=pos_prompt,
    neg_prompt=neg_prompt,
    cfg_scale=cfg_scale,
    better_start=better_start,
    capture_trace=True,
    progress=True,
)

df = rows_to_dataframe(rows)
display(df)

## Step Degradation Grids

Each grid is saved to `figures/{image_id}_step_grid.png`.

In [ ]:
for pair in pairs:
    input_image = load_rgb(pair.input_path)
    target_image = load_rgb(pair.target_path) if pair.target_path else None
    restored_by_step = load_restored_by_step(output_dir, pair.image_id, steps_list)
    fig = plot_step_grid(
        image_id=pair.image_id,
        input_image=input_image,
        target_image=target_image,
        restored_by_step=restored_by_step,
        steps_list=steps_list,
        save_path=dirs["figures"] / f"{pair.image_id}_step_grid.png",
    )
    display(fig)
    plt.close(fig)

## Runtime-Quality Curve

The plot is saved to `figures/runtime_quality_curve.png`.

In [ ]:
fig = plot_runtime_quality(rows, save_path=dirs["figures"] / "runtime_quality_curve.png")
display(fig)
plt.close(fig)

## Internal Dynamics Heatmaps

Use low-step outputs first because they are the most useful motivation examples. Each figure includes EAC residual, epsilon temporal change, `pred_x0` temporal change, and a hard-region overlay.

In [ ]:
probe_steps = [10, 8, 5]
for pair in pairs:
    input_image = load_rgb(pair.input_path)
    target_image = load_rgb(pair.target_path) if pair.target_path else None
    for steps in probe_steps:
        trace_path = dirs["tensors"] / f"{pair.image_id}_steps{steps}_trace.npz"
        restored_path = dirs["restored"] / f"{pair.image_id}_steps{steps}.png"
        if not trace_path.exists() or not restored_path.exists():
            continue
        restored_image = load_rgb(restored_path)
        fig = plot_internal_maps(
            trace_path=trace_path,
            input_image=input_image,
            target_image=target_image,
            restored_image=restored_image,
            save_path=dirs["figures"] / f"{pair.image_id}_steps{steps}_internal_maps.png",
        )
        fig.suptitle(f"{pair.image_id}, {steps} steps")
        display(fig)
        plt.close(fig)

## Optional PyIQA Environment Check

Run this section in the separate IQA environment. Do not install `pyiqa` into the DeblurDiff inference environment unless dependencies are pinned.

In [ ]:
try:
    print(check_pyiqa_environment())
except Exception as exc:
    print(f"pyiqa is not ready in this kernel: {exc}")

In [ ]:
# Optional sanity check. These metrics may download pretrained weights on first use.
metric_names = ["lpips", "niqe", "musiq", "maniqa", "topiq_nr"]
if pairs:
    pair = pairs[0]
    restored_path = dirs["restored"] / f"{pair.image_id}_steps10.png"
    for metric_name in metric_names:
        try:
            if metric_name in {"lpips"} and pair.target_path is not None:
                score = run_pyiqa_pair(metric_name, restored_path, pair.target_path, device=device)
            else:
                score = run_pyiqa_pair(metric_name, restored_path, None, device=device)
            print(metric_name, score)
        except Exception as exc:
            print(f"{metric_name}: skipped ({exc})")